# AquaVanta Pulse 750: Synthetic Omnichannel Sales Dataset Generator

The notebook exports clean relational tables, a flattened machine-learning table, a
deliberately messy data-cleaning edition, metadata, validation summaries, and QA charts.

> All brands, events, transactions, and operational records are fictional. The dataset
> must not be represented as observed commercial data.

## 1. Environment and reproducibility

A fixed random seed makes every run reproducible. Change `RANDOM_SEED` to create a new,
statistically comparable edition.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# A single seeded generator is used throughout the notebook so the output is deterministic.
RANDOM_SEED = 750
rng = np.random.default_rng(RANDOM_SEED)

START_DATE = "2023-01-01"
END_DATE = "2025-12-31"
OUTPUT_ROOT = Path("aquavanta_pulse_dataset")
CLEAN_DIR = OUTPUT_ROOT / "clean"
RAW_DIR = OUTPUT_ROOT / "raw"
METADATA_DIR = OUTPUT_ROOT / "metadata"
FIGURE_DIR = OUTPUT_ROOT / "figures"

for directory in [CLEAN_DIR, RAW_DIR, METADATA_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)

print(f"Random seed: {RANDOM_SEED}")
print(f"Output directory: {OUTPUT_ROOT.resolve()}")

## 2. Market, channel, and product configuration

In [ ]:
# City factors affect traffic and demand. Climate parameters create regionally different
# temperature, rainfall, humidity, and air-quality patterns.
city_config = pd.DataFrame(
    [
        ("DEL", "Delhi NCR", 1.25, 30.0, 10.0, 0.70, 50, 165, 1.0),
        ("MUM", "Mumbai", 1.30, 29.5, 3.5, 1.55, 72, 90, 0.5),
        ("BLR", "Bengaluru", 1.15, 26.5, 2.8, 0.90, 62, 75, 0.8),
        ("HYD", "Hyderabad", 1.00, 30.5, 5.5, 0.70, 55, 95, 1.0),
        ("CHE", "Chennai", 0.95, 30.5, 3.0, 1.20, 73, 85, 1.2),
        ("KOL", "Kolkata", 0.90, 29.5, 5.0, 1.25, 69, 120, 1.3),
        ("PUN", "Pune", 0.88, 28.0, 4.0, 0.90, 58, 80, 0.9),
        ("AMD", "Ahmedabad", 0.82, 31.5, 8.0, 0.58, 45, 105, 1.1),
    ],
    columns=[
        "city_id", "city", "city_demand_factor", "annual_temp_mean",
        "temp_amplitude", "rain_factor", "base_humidity", "base_aqi",
        "lead_time_adjustment",
    ],
)

# Channels intentionally trade volume, margin, marketing efficiency, delivery speed,
# and returns differently, as real omnichannel businesses do.
channel_config = pd.DataFrame(
    [
        ("D2C", "Brand Website", 0.95, 1.00, 0.040, 4_500, 180, 3.0, 0.00),
        ("MKT", "Marketplace", 1.35, 1.15, 0.070, 3_200, 140, 2.5, 0.18),
        ("RTL", "Physical Retail", 0.72, 0.72, 0.030, 1_500, 260, 0.2, 0.08),
    ],
    columns=[
        "channel_id", "channel", "channel_demand_factor", "traffic_factor",
        "base_discount", "base_channel_ad_spend", "base_cpm",
        "base_delivery_days", "channel_fee_rate",
    ],
)

# Variants belong to one product family. Their preference factors differ by season and
# channel later in the notebook rather than remaining constant everywhere.
variant_config = pd.DataFrame(
    [
        ("AVP-GRA", "Graphite", 1.15, 0.28, 940),
        ("AVP-OCN", "Ocean Blue", 1.08, 0.27, 950),
        ("AVP-COR", "Coral", 0.88, 0.22, 955),
        ("AVP-SAG", "Sage", 0.92, 0.23, 952),
    ],
    columns=["sku", "colour_variant", "variant_demand_factor", "ad_allocation_share", "base_unit_cost"],
)

print("Cities:", len(city_config), "| Channels:", len(channel_config), "| Variants:", len(variant_config))
print(variant_config.to_string(index=False))

## 3. Event catalogue

In [ ]:
events = pd.DataFrame(
    [
        ("EVT001", "Product launch burst", "2023-01-01", "2023-01-21", "All", "Demand/Traffic", "Launch publicity creates a strong but decaying traffic burst."),
        ("EVT002", "Marketplace listing outage", "2023-12-12", "2023-12-12", "Marketplace", "Traffic", "A partial listing outage suppresses marketplace visits."),
        ("EVT003", "Influencer campaign", "2024-04-15", "2024-04-24", "Delhi, Mumbai, Bengaluru D2C", "Marketing", "A creator campaign increases D2C traffic and discounting."),
        ("EVT004", "North-west heatwave", "2024-05-10", "2024-05-25", "Delhi, Ahmedabad", "Weather/Demand", "Exceptional heat increases hydration-product demand."),
        ("EVT005", "Supplier component delay", "2024-06-10", "2024-06-30", "All", "Supply", "A delayed electronic cap component increases replenishment lead time."),
        ("EVT006", "Battery-cap quality issue", "2024-08-01", "2024-08-31", "Coral, Sage", "Quality", "A defective production batch raises returns and lowers review scores."),
        ("EVT007", "Targeted product recall", "2024-09-01", "2024-09-15", "Coral, Sage", "Demand/Quality", "A transparent recall temporarily reduces demand for affected colours."),
        ("EVT008", "Western courier strike", "2025-07-10", "2025-07-20", "Mumbai, Pune", "Delivery", "Courier capacity falls, causing delays and cancellations."),
        ("EVT009", "Competitor product launch", "2025-12-01", "2025-12-31", "All", "Competition", "A lower-priced competitor puts pressure on conversion."),
        ("EVT010", "Counter-promotion", "2025-12-15", "2025-12-31", "All", "Pricing/Marketing", "AquaVanta responds with a time-limited promotion."),
    ],
    columns=["event_id", "event_name", "start_date", "end_date", "scope", "event_type", "description"],
)
events["start_date"] = pd.to_datetime(events["start_date"])
events["end_date"] = pd.to_datetime(events["end_date"])

print(events[["event_id", "event_name", "start_date", "end_date", "scope"]].to_string(index=False))

## 4. Construct the daily observation grid

In [ ]:
dates = pd.date_range(START_DATE, END_DATE, freq="D")

# A Cartesian product guarantees complete coverage before event effects or missingness are added.
grid = pd.MultiIndex.from_product(
    [dates, city_config["city_id"], channel_config["channel_id"], variant_config["sku"]],
    names=["date", "city_id", "channel_id", "sku"],
).to_frame(index=False)

df = (
    grid
    .merge(city_config, on="city_id", how="left", validate="many_to_one")
    .merge(channel_config, on="channel_id", how="left", validate="many_to_one")
    .merge(variant_config, on="sku", how="left", validate="many_to_one")
    .sort_values(["city_id", "channel_id", "sku", "date"])
    .reset_index(drop=True)
)

# Stable record identifiers make it possible to trace rows across clean and raw editions.
df.insert(0, "record_id", [f"AVP-{i:06d}" for i in range(1, len(df) + 1)])
df["days_since_launch"] = (df["date"] - dates.min()).dt.days
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.day_name()
df["is_weekend"] = df["date"].dt.dayofweek.ge(4)
df["is_payday_window"] = df["date"].dt.day.le(5) | df["date"].dt.day.ge(28)

expected_rows = len(dates) * len(city_config) * len(channel_config) * len(variant_config)
assert len(df) == expected_rows == 105_216
print(f"Created {len(df):,} observations from {dates.min().date()} to {dates.max().date()}.")

## 5. Calendar and regional weather

In [ ]:
# Festival dates are explicit because major Indian festivals do not occur on a fixed
# Gregorian date. Windows are wider for Diwali to represent the extended sale period.
festival_dates = {
    "Holi": ["2023-03-08", "2024-03-25", "2025-03-14"],
    "Eid": ["2023-04-22", "2024-04-11", "2025-03-31"],
    "Diwali": ["2023-11-12", "2024-11-01", "2025-10-20"],
    "Christmas": ["2023-12-25", "2024-12-25", "2025-12-25"],
}

# Use an explicit label rather than the literal string "None", which pandas may parse as
# a missing value when users later read the exported CSV with default settings.
df["festival_name"] = "No festival"
for festival, festival_day_list in festival_dates.items():
    for festival_day in pd.to_datetime(festival_day_list):
        before, after = (7, 2) if festival == "Diwali" else (2, 1)
        mask = df["date"].between(festival_day - pd.Timedelta(days=before), festival_day + pd.Timedelta(days=after))
        df.loc[mask, "festival_name"] = festival

fixed_holidays = []
for year in [2023, 2024, 2025]:
    fixed_holidays.extend(pd.to_datetime([f"{year}-01-26", f"{year}-08-15", f"{year}-10-02"]))
df["is_public_holiday"] = df["date"].isin(fixed_holidays) | df["festival_name"].ne("No festival")

day_of_year = df["date"].dt.dayofyear.to_numpy()
seasonal_wave = np.cos(2 * np.pi * (day_of_year - 135) / 365.25)

# Correlated weather noise is generated at the city-date level and then merged, avoiding
# impossible differences between channels or colours in the same city on the same day.
weather_grid = df[["date", "city_id", "city", "annual_temp_mean", "temp_amplitude", "rain_factor", "base_humidity", "base_aqi"]].drop_duplicates().copy()
weather_doy = weather_grid["date"].dt.dayofyear.to_numpy()
weather_wave = np.cos(2 * np.pi * (weather_doy - 135) / 365.25)
weather_grid["maximum_temperature_c"] = (
    weather_grid["annual_temp_mean"].to_numpy()
    + weather_grid["temp_amplitude"].to_numpy() * weather_wave
    + rng.normal(0, 1.8, len(weather_grid))
)

monsoon = weather_grid["date"].dt.month.isin([6, 7, 8, 9]).to_numpy()
chennai_retreating_monsoon = weather_grid["city_id"].eq("CHE").to_numpy() & weather_grid["date"].dt.month.isin([10, 11]).to_numpy()
rain_probability = np.where(monsoon, 0.30 * weather_grid["rain_factor"].to_numpy(), 0.035)
rain_probability = np.where(chennai_retreating_monsoon, 0.48, rain_probability)
rain_occurs = rng.random(len(weather_grid)) < np.clip(rain_probability, 0, 0.80)
weather_grid["rainfall_mm"] = np.where(
    rain_occurs,
    rng.gamma(shape=1.8, scale=6.0 * weather_grid["rain_factor"].to_numpy()),
    0.0,
)
weather_grid["humidity_pct"] = np.clip(
    weather_grid["base_humidity"].to_numpy()
    + 12 * rain_occurs
    - 0.45 * (weather_grid["maximum_temperature_c"].to_numpy() - 30)
    + rng.normal(0, 5, len(weather_grid)),
    25,
    98,
)

winter = weather_grid["date"].dt.month.isin([11, 12, 1]).to_numpy()
delhi_winter_penalty = winter * weather_grid["city_id"].eq("DEL").to_numpy() * 95
weather_grid["air_quality_index"] = np.clip(
    weather_grid["base_aqi"].to_numpy() + winter * 28 + delhi_winter_penalty + rng.normal(0, 22, len(weather_grid)),
    25,
    450,
)

df = df.merge(
    weather_grid[["date", "city_id", "maximum_temperature_c", "rainfall_mm", "humidity_pct", "air_quality_index"]],
    on=["date", "city_id"],
    how="left",
    validate="many_to_one",
)

# Apply the exceptional 2024 heatwave after normal weather is created.
heatwave_event = df["date"].between("2024-05-10", "2024-05-25") & df["city_id"].isin(["DEL", "AMD"])
df.loc[heatwave_event, "maximum_temperature_c"] += 4.0
df["heatwave_flag"] = df["maximum_temperature_c"].ge(40) | heatwave_event

for col in ["maximum_temperature_c", "rainfall_mm", "humidity_pct", "air_quality_index"]:
    df[col] = df[col].round(2)

print(df[["date", "city", "maximum_temperature_c", "rainfall_mm", "humidity_pct"]].sample(5, random_state=RANDOM_SEED).to_string(index=False))

## 6. Price, competition, campaigns, and event flags

In [ ]:
# Start every multiplicative event variable at its neutral value.
df["event_name"] = "No event"
df["traffic_event_multiplier"] = 1.0
df["demand_event_multiplier"] = 1.0
df["lead_time_extra_days"] = 0.0
df["delivery_delay_extra_days"] = 0.0
df["quality_issue_flag"] = False
df["campaign_type"] = "Always-on"

def label_event(mask, name):
    # Append an event label without discarding another overlapping event.
    existing = df.loc[mask, "event_name"]
    df.loc[mask, "event_name"] = np.where(existing.eq("No event"), name, existing + " | " + name)

launch = df["date"].between("2023-01-01", "2023-01-21")
outage = df["date"].eq("2023-12-12") & df["channel_id"].eq("MKT")
influencer = (
    df["date"].between("2024-04-15", "2024-04-24")
    & df["city_id"].isin(["DEL", "MUM", "BLR"])
    & df["channel_id"].eq("D2C")
)
supplier_delay = df["date"].between("2024-06-10", "2024-06-30")
defect = df["date"].between("2024-08-01", "2024-08-31") & df["sku"].isin(["AVP-COR", "AVP-SAG"])
recall = df["date"].between("2024-09-01", "2024-09-15") & df["sku"].isin(["AVP-COR", "AVP-SAG"])
courier_strike = df["date"].between("2025-07-10", "2025-07-20") & df["city_id"].isin(["MUM", "PUN"])
competitor_launch = df["date"].between("2025-12-01", "2025-12-31")
counter_promotion = df["date"].between("2025-12-15", "2025-12-31")

# A launch effect decays through the first three weeks rather than stopping abruptly.
launch_decay = np.exp(-df.loc[launch, "days_since_launch"].to_numpy() / 18)
df.loc[launch, "traffic_event_multiplier"] *= 1.25 + 0.35 * launch_decay
df.loc[launch, "demand_event_multiplier"] *= 1.12 + 0.18 * launch_decay
label_event(launch, "Product launch burst")

df.loc[outage, "traffic_event_multiplier"] *= 0.35
label_event(outage, "Marketplace listing outage")

df.loc[influencer, "traffic_event_multiplier"] *= 1.85
df.loc[influencer, "demand_event_multiplier"] *= 1.18
df.loc[influencer, "campaign_type"] = "Influencer"
label_event(influencer, "Influencer campaign")

label_event(heatwave_event, "North-west heatwave")
df.loc[supplier_delay, "lead_time_extra_days"] = 7.0
label_event(supplier_delay, "Supplier component delay")

df.loc[defect, "quality_issue_flag"] = True
label_event(defect, "Battery-cap quality issue")
df.loc[recall, "demand_event_multiplier"] *= 0.64
label_event(recall, "Targeted product recall")

df.loc[courier_strike, "delivery_delay_extra_days"] = 3.5
label_event(courier_strike, "Western courier strike")
label_event(competitor_launch, "Competitor product launch")
df.loc[counter_promotion, "campaign_type"] = "Counter-promotion"
df.loc[counter_promotion, "traffic_event_multiplier"] *= 1.20
label_event(counter_promotion, "Counter-promotion")

# List price rises moderately in 2025. Discounts are built from several observable causes,
# then rounded to common commercial increments and capped at 30%.
df["list_price"] = np.where(df["date"].ge("2025-01-01"), 2_599.0, 2_499.0)
coupon_noise = rng.choice([0.00, 0.02, 0.05], len(df), p=[0.67, 0.25, 0.08])
festival_discount = np.select(
    [df["festival_name"].eq("Diwali"), df["festival_name"].isin(["Holi", "Eid", "Christmas"])],
    [0.10, 0.05],
    default=0.0,
)
df["discount_pct"] = (
    df["base_discount"]
    + 0.015 * df["is_weekend"].astype(float)
    + festival_discount
    + coupon_noise
    + 0.08 * influencer.astype(float)
    + 0.12 * counter_promotion.astype(float)
).clip(0, 0.30).round(2)
df["selling_price"] = (df["list_price"] * (1 - df["discount_pct"])).round(2)

# Competitor prices drift with inflation and seasonal promotions. The December 2025 launch
# creates an explicit price shock that AquaVanta only partly offsets.
competitor_inflation = 1 + 0.035 * (df["year"] - 2023)
competitor_promo = np.where(df["festival_name"].ne("No festival"), 0.94, 1.0)
competitor_shock = np.where(competitor_launch, 0.82, 1.0)
df["competitor_median_price"] = (
    2_399 * competitor_inflation * competitor_promo * competitor_shock
    + rng.normal(0, 55, len(df))
).clip(1_650, 2_850).round(2)
df["price_gap_pct"] = ((df["selling_price"] - df["competitor_median_price"]) / df["competitor_median_price"]).round(4)

print(df[["date", "channel", "festival_name", "campaign_type", "discount_pct", "selling_price", "competitor_median_price", "event_name"]].sample(8, random_state=21).to_string(index=False))

## 7. Reputation, advertising, and the shopping funnel

In [ ]:
# Expected reputation is slightly channel-dependent and temporarily harmed by the defect.
df["expected_rating"] = 4.34 + df["channel_id"].map({"D2C": 0.09, "MKT": -0.05, "RTL": 0.03}).astype(float)
df.loc[defect, "expected_rating"] -= 0.95

# Reputation remains impaired after the defective batch, then recovers over roughly 90 days.
recovery_mask = df["date"].between("2024-09-01", "2024-11-30") & df["sku"].isin(["AVP-COR", "AVP-SAG"])
recovery_days = (df.loc[recovery_mask, "date"] - pd.Timestamp("2024-09-01")).dt.days.to_numpy()
df.loc[recovery_mask, "expected_rating"] -= 0.75 * np.exp(-recovery_days / 35)
df["expected_rating"] = df["expected_rating"].clip(2.8, 4.7)

group_keys = ["city_id", "channel_id", "sku"]
df["lagged_reputation"] = (
    df.groupby(group_keys, sort=False)["expected_rating"]
    .transform(lambda s: s.rolling(21, min_periods=1).mean().shift(7))
    .fillna(4.30)
)

# Brand awareness grows quickly after launch and slowly approaches a plateau.
brand_growth = 0.72 + 0.58 * (1 - np.exp(-df["days_since_launch"].to_numpy() / 520))
summer_marketing = 1 + 0.15 * df["month"].isin([3, 4, 5, 6]).astype(float)
festival_marketing = 1 + 0.40 * df["festival_name"].ne("No festival").astype(float)
campaign_marketing = np.where(influencer, 2.20, np.where(counter_promotion, 1.55, 1.0))

# Ad spend is a channel-city total allocated to variants using explicit budget shares.
df["allocated_ad_spend"] = (
    df["base_channel_ad_spend"]
    * df["city_demand_factor"]
    * df["ad_allocation_share"]
    * summer_marketing
    * festival_marketing
    * campaign_marketing
    * rng.lognormal(mean=0, sigma=0.10, size=len(df))
).round(2)

cpm = df["base_cpm"].to_numpy() * rng.lognormal(0, 0.08, len(df))
df["impressions"] = np.rint(df["allocated_ad_spend"].to_numpy() / cpm * 1_000).astype(int)
ctr = (
    0.012
    + 0.005 * df["campaign_type"].isin(["Influencer", "Counter-promotion"]).astype(float)
    + 0.002 * df["festival_name"].ne("No festival").astype(float)
    + rng.normal(0, 0.0015, len(df))
).clip(0.004, 0.035)
df["clicks"] = rng.binomial(df["impressions"].to_numpy(), ctr.to_numpy())

# Organic visits and paid clicks combine into total product-page traffic. Retail uses the
# same variable as a digital-assisted footfall proxy, which is documented in the dictionary.
ocean_summer_bonus = np.where((df["sku"].eq("AVP-OCN")) & df["month"].isin([3, 4, 5, 6]), 1.18, 1.0)
graphite_retail_bonus = np.where((df["sku"].eq("AVP-GRA")) & df["channel_id"].eq("RTL"), 1.12, 1.0)
organic_views = (
    105
    * df["city_demand_factor"].to_numpy()
    * df["traffic_factor"].to_numpy()
    * df["variant_demand_factor"].to_numpy()
    * brand_growth
    * ocean_summer_bonus
    * graphite_retail_bonus
    * df["traffic_event_multiplier"].to_numpy()
    * rng.lognormal(0, 0.16, len(df))
)
paid_visit_rate = np.where(df["channel_id"].eq("RTL"), 0.18, 0.72)
df["product_page_views"] = np.rint(organic_views + df["clicks"].to_numpy() * paid_visit_rate).clip(1).astype(int)

# Price, discounts, weather, reputation, and calendar effects influence conversion.
heat_effect = np.clip(df["maximum_temperature_c"].to_numpy() - 28, 0, 15) * 0.0015
price_pressure = np.clip(df["price_gap_pct"].to_numpy(), -0.30, 0.35)
cart_rate = (
    0.070
    + 0.16 * df["discount_pct"].to_numpy()
    + heat_effect
    + 0.010 * df["is_payday_window"].to_numpy()
    + 0.012 * df["is_public_holiday"].to_numpy()
    + 0.012 * (df["lagged_reputation"].to_numpy() - 4.0)
    - 0.035 * np.maximum(price_pressure, 0)
    + rng.normal(0, 0.006, len(df))
)
cart_rate = np.clip(cart_rate, 0.035, 0.18)
df["add_to_cart_count"] = rng.binomial(df["product_page_views"].to_numpy(), cart_rate)

# Heavy rainfall diverts some demand from stores to online channels.
rain_mm = df["rainfall_mm"].to_numpy()
retail_rain_effect = np.where(df["channel_id"].eq("RTL"), np.clip(1 - 0.012 * rain_mm, 0.60, 1.0), 1.0)
online_rain_effect = np.where(df["channel_id"].isin(["D2C", "MKT"]), 1 + np.clip(0.0025 * rain_mm, 0, 0.12), 1.0)
competitor_effect = np.exp(-0.65 * np.maximum(price_pressure, 0))
reputation_effect = np.clip((df["lagged_reputation"].to_numpy() / 4.30) ** 2.0, 0.65, 1.15)

checkout_rate = (
    0.30
    + 0.30 * df["discount_pct"].to_numpy()
    + 0.02 * df["is_weekend"].to_numpy()
    + 0.03 * df["is_payday_window"].to_numpy()
    + rng.normal(0, 0.018, len(df))
)
checkout_rate *= (
    retail_rain_effect
    * online_rain_effect
    * competitor_effect
    * reputation_effect
    * df["demand_event_multiplier"].to_numpy()
)
checkout_rate = np.clip(checkout_rate, 0.16, 0.65)
df["potential_demand"] = rng.binomial(df["add_to_cart_count"].to_numpy(), checkout_rate)
df["units_ordered"] = df["potential_demand"].astype(int)
df["conversion_rate"] = (df["potential_demand"] / df["product_page_views"]).round(4)

assert (df["clicks"] <= df["impressions"]).all()
assert (df["add_to_cart_count"] <= df["product_page_views"]).all()
assert (df["potential_demand"] <= df["add_to_cart_count"]).all()

print(df[["impressions", "clicks", "product_page_views", "add_to_cart_count", "potential_demand", "conversion_rate"]].describe().round(2))

## 8. Sequential inventory and fulfilment simulation

In [ ]:
# Preallocate arrays because writing into NumPy arrays is substantially faster than
# repeatedly assigning individual DataFrame cells inside the simulation loop.
n_rows = len(df)
opening_inventory = np.zeros(n_rows, dtype=int)
replenishment_units = np.zeros(n_rows, dtype=int)
order_placed_units = np.zeros(n_rows, dtype=int)
closing_inventory = np.zeros(n_rows, dtype=int)
cancellation_units = np.zeros(n_rows, dtype=int)
units_sold = np.zeros(n_rows, dtype=int)
lost_sales_units = np.zeros(n_rows, dtype=int)
stockout_hours = np.zeros(n_rows, dtype=float)
supplier_lead_time_days = np.zeros(n_rows, dtype=float)

# A child generator isolates the inventory process while preserving full reproducibility.
inventory_rng = np.random.default_rng(RANDOM_SEED + 101)

for _, group in df.groupby(group_keys, sort=False):
    idx = group.index.to_numpy()
    demand_series = group["potential_demand"].to_numpy(dtype=int)
    base_lead = 6.0 + float(group["lead_time_adjustment"].iloc[0])
    base_lead += {"D2C": 1.0, "MKT": -1.0, "RTL": 3.0}[group["channel_id"].iloc[0]]

    # Initial stock covers about 18 days of early demand, with cross-location variation.
    early_average = max(float(np.mean(demand_series[:21])), 1.0)
    stock = int(np.ceil(early_average * inventory_rng.uniform(15, 21)))
    forecast = early_average
    pending_arrivals = {}

    for local_day, row_index in enumerate(idx):
        arrivals = int(pending_arrivals.pop(local_day, 0))
        opening_inventory[row_index] = stock
        replenishment_units[row_index] = arrivals
        stock += arrivals

        extra_lead = float(df.at[row_index, "lead_time_extra_days"])
        actual_lead = max(2, int(round(base_lead + extra_lead + inventory_rng.normal(0, 1.2))))
        supplier_lead_time_days[row_index] = actual_lead

        # Expected delivery problems raise pre-fulfilment cancellation probability.
        cancel_probability = 0.018 + 0.012 * (df.at[row_index, "channel_id"] == "MKT")
        cancel_probability += 0.065 * (df.at[row_index, "delivery_delay_extra_days"] > 0)
        cancel_probability += min(df.at[row_index, "rainfall_mm"] / 600, 0.035)
        cancellations = inventory_rng.binomial(demand_series[local_day], min(cancel_probability, 0.18))
        cancellation_units[row_index] = cancellations

        fulfilment_demand = max(demand_series[local_day] - cancellations, 0)
        sales = min(stock, fulfilment_demand)
        lost = fulfilment_demand - sales
        stock -= sales

        units_sold[row_index] = sales
        lost_sales_units[row_index] = lost
        closing_inventory[row_index] = stock
        stockout_hours[row_index] = 0 if lost == 0 else min(24, 24 * lost / max(fulfilment_demand, 1))

        # Exponentially weighted demand updates slowly enough that sudden successful
        # campaigns can still cause plausible short-term stockouts.
        forecast = 0.88 * forecast + 0.12 * max(demand_series[local_day], 0.5)
        inventory_position = stock + sum(pending_arrivals.values())
        reorder_point = forecast * (actual_lead + 4)
        target_position = forecast * (actual_lead + 18)

        if inventory_position < reorder_point:
            order_quantity = max(int(math.ceil(target_position - inventory_position)), 0)
            order_placed_units[row_index] = order_quantity
            arrival_day = local_day + actual_lead
            if arrival_day < len(idx):
                pending_arrivals[arrival_day] = pending_arrivals.get(arrival_day, 0) + order_quantity

df["opening_inventory"] = opening_inventory
df["replenishment_units"] = replenishment_units
df["order_placed_units"] = order_placed_units
df["closing_inventory"] = closing_inventory
df["cancellation_units"] = cancellation_units
df["units_sold"] = units_sold
df["lost_sales_units"] = lost_sales_units
df["stockout_hours"] = stockout_hours.round(2)
df["supplier_lead_time_days"] = supplier_lead_time_days.round(1)

# Inventory cover uses only prior 14-day sales, preventing look-ahead leakage.
trailing_sales = df.groupby(group_keys, sort=False)["units_sold"].transform(
    lambda s: s.shift(1).rolling(14, min_periods=3).mean()
)
df["inventory_cover_days"] = (
    df["closing_inventory"] / trailing_sales.replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(30).clip(0, 90).round(2)

assert (df["units_sold"] <= df["units_ordered"] - df["cancellation_units"]).all()
assert (
    df["opening_inventory"] + df["replenishment_units"] - df["units_sold"] == df["closing_inventory"]
).all()

print("Fulfilled units:", f"{df['units_sold'].sum():,}")
print("Lost sales units:", f"{df['lost_sales_units'].sum():,}")
print("Rows with a stockout:", f"{df['stockout_hours'].gt(0).mean():.2%}")

## 9. Delivery, delayed returns, reviews, and economics

In [ ]:
# Delivery time responds to channel, rain, regional distance, and the courier strike.
delivery_noise = rng.gamma(shape=1.5, scale=0.35, size=len(df))
df["average_delivery_days"] = np.clip(
    df["base_delivery_days"].to_numpy()
    + 0.35 * df["lead_time_adjustment"].to_numpy()
    + 0.018 * df["rainfall_mm"].to_numpy()
    + df["delivery_delay_extra_days"].to_numpy()
    + delivery_noise,
    0,
    12,
).round(2)

# Shift purchase attributes forward so today's returns represent units bought 14 days ago.
lagged_sales_14 = df.groupby(group_keys, sort=False)["units_sold"].shift(14).fillna(0).astype(int)
lagged_discount_14 = df.groupby(group_keys, sort=False)["discount_pct"].shift(14).fillna(0)
lagged_delivery_14 = df.groupby(group_keys, sort=False)["average_delivery_days"].shift(14).fillna(0)
lagged_defect_14 = df.groupby(group_keys, sort=False)["quality_issue_flag"].shift(14).fillna(False).astype(bool)
lagged_price_14 = df.groupby(group_keys, sort=False)["selling_price"].shift(14).fillna(df["selling_price"])

return_probability = (
    0.038
    + 0.020 * df["channel_id"].eq("MKT").to_numpy()
    - 0.010 * df["channel_id"].eq("RTL").to_numpy()
    + 0.14 * lagged_discount_14.to_numpy()
    + 0.014 * np.clip(lagged_delivery_14.to_numpy() - 4, 0, None)
    + 0.20 * lagged_defect_14.to_numpy()
)
return_probability = np.clip(return_probability, 0.01, 0.38)
df["return_units"] = rng.binomial(lagged_sales_14.to_numpy(), return_probability)
df["return_rate"] = (df["return_units"] / lagged_sales_14.replace(0, np.nan)).fillna(0).round(4)

# Reviews arrive slightly earlier than most returns. Average ratings are noisier when only
# one or two reviews are posted and more stable when review volume is high.
lagged_sales_10 = df.groupby(group_keys, sort=False)["units_sold"].shift(10).fillna(0).astype(int)
lagged_rating_10 = df.groupby(group_keys, sort=False)["expected_rating"].shift(10).fillna(4.3)
lagged_delivery_10 = df.groupby(group_keys, sort=False)["average_delivery_days"].shift(10).fillna(0)
lagged_defect_10 = df.groupby(group_keys, sort=False)["quality_issue_flag"].shift(10).fillna(False).astype(bool)

review_probability = np.clip(0.11 + 0.04 * lagged_defect_10.to_numpy(), 0.05, 0.22)
df["new_review_count"] = rng.binomial(lagged_sales_10.to_numpy(), review_probability)
review_mean = (
    lagged_rating_10.to_numpy()
    - 0.10 * np.clip(lagged_delivery_10.to_numpy() - 4, 0, None)
    - 0.18 * lagged_defect_10.to_numpy()
)
review_sd = 0.65 / np.sqrt(np.maximum(df["new_review_count"].to_numpy(), 1))
daily_rating = np.clip(rng.normal(review_mean, review_sd), 1.0, 5.0)
df["average_rating"] = np.where(df["new_review_count"].gt(0), daily_rating, np.nan).round(2)

negative_probability = np.clip(0.04 + 0.18 * (4.2 - review_mean) + 0.16 * lagged_defect_10.to_numpy(), 0.01, 0.75)
negative_reviews = rng.binomial(df["new_review_count"].to_numpy(), negative_probability)
df["negative_review_share"] = (
    negative_reviews / df["new_review_count"].replace(0, np.nan)
).fillna(0).round(4)

# The dominant return reason follows the conditions attached to the original purchase.
default_reasons = rng.choice(
    ["Changed mind", "Colour mismatch", "Expectation mismatch", "Damaged in transit"],
    len(df),
    p=[0.42, 0.24, 0.24, 0.10],
)
df["primary_return_reason"] = np.where(
    df["return_units"].eq(0),
    "No return",
    np.where(
        lagged_defect_14,
        "Battery-cap defect",
        np.where(lagged_delivery_14.gt(5), "Late delivery", np.where(lagged_discount_14.gt(0.20), "Changed mind", default_reasons)),
    ),
)

# Costs rise modestly over time. Net revenue recognizes today's returns from earlier sales,
# so individual days can be negative after a major promotion or quality incident.
df["unit_cost"] = (
    df["base_unit_cost"] * (1 + 0.035 * (df["year"] - 2023))
).round(2)
df["gross_revenue"] = (df["units_sold"] * df["selling_price"]).round(2)
df["return_value"] = (df["return_units"] * lagged_price_14).round(2)
df["net_revenue"] = (df["gross_revenue"] - df["return_value"]).round(2)
channel_fees = df["gross_revenue"] * df["channel_fee_rate"]
df["gross_margin"] = (
    df["net_revenue"]
    - df["units_sold"] * df["unit_cost"]
    - channel_fees
    - df["allocated_ad_spend"]
    - df["return_units"] * 150
).round(2)

print(df[["units_sold", "average_delivery_days", "return_units", "return_rate", "new_review_count", "average_rating", "net_revenue", "gross_margin"]].describe().round(2))

## 10. Assemble relational and model-ready tables

In [ ]:
# Round the remaining continuous values before export for compact, readable CSV files.
df["allocated_ad_spend"] = df["allocated_ad_spend"].round(2)
df["expected_rating"] = df["expected_rating"].round(3)
df["lagged_reputation"] = df["lagged_reputation"].round(3)

identifier_cols = ["record_id", "date", "city_id", "city", "channel_id", "channel", "sku", "colour_variant"]
calendar_cols = ["year", "month", "day_of_week", "days_since_launch", "is_weekend", "is_payday_window", "is_public_holiday", "festival_name"]
weather_cols = ["maximum_temperature_c", "rainfall_mm", "humidity_pct", "air_quality_index", "heatwave_flag"]
price_cols = ["list_price", "discount_pct", "selling_price", "competitor_median_price", "price_gap_pct", "unit_cost"]
marketing_cols = ["campaign_type", "allocated_ad_spend", "impressions", "clicks", "product_page_views", "add_to_cart_count"]
outcome_cols = [
    "potential_demand", "units_ordered", "cancellation_units", "units_sold", "lost_sales_units",
    "conversion_rate", "gross_revenue", "return_value", "net_revenue", "gross_margin",
    "average_delivery_days", "return_units", "return_rate", "new_review_count",
    "average_rating", "negative_review_share", "primary_return_reason", "event_name",
]
inventory_cols = [
    "opening_inventory", "replenishment_units", "order_placed_units", "closing_inventory",
    "stockout_hours", "inventory_cover_days", "supplier_lead_time_days",
]

sales_daily = df[identifier_cols + calendar_cols + price_cols + marketing_cols + outcome_cols].copy()
inventory_daily = df[identifier_cols + inventory_cols].copy()

# Weather is one record per city-date, independent of product variant and sales channel.
weather_calendar = (
    df[["date", "city_id", "city"] + weather_cols + ["is_public_holiday", "festival_name"]]
    .drop_duplicates(["date", "city_id"])
    .sort_values(["date", "city_id"])
    .reset_index(drop=True)
)

# Marketing spend and funnel counts sum correctly because allocated variant budgets are used.
marketing_daily = (
    df.groupby(["date", "city_id", "city", "channel_id", "channel", "campaign_type"], as_index=False)
    .agg(
        ad_spend=("allocated_ad_spend", "sum"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        product_page_views=("product_page_views", "sum"),
        add_to_cart_count=("add_to_cart_count", "sum"),
    )
)
marketing_daily["ad_spend"] = marketing_daily["ad_spend"].round(2)

returns_reviews = df.loc[
    df["return_units"].gt(0) | df["new_review_count"].gt(0),
    identifier_cols + [
        "return_units", "return_rate", "return_value", "primary_return_reason",
        "new_review_count", "average_rating", "negative_review_share", "average_delivery_days",
    ],
].copy()

# The flattened table keeps all predictive variables but excludes internal simulation-only
# multipliers that would make modelling unrealistically easy.
model_ready_cols = (
    identifier_cols + calendar_cols + weather_cols + price_cols + marketing_cols
    + inventory_cols + outcome_cols
)
model_ready_daily = df[model_ready_cols].copy()

print("sales_daily:", sales_daily.shape)
print("inventory_daily:", inventory_daily.shape)
print("marketing_daily:", marketing_daily.shape)
print("weather_calendar:", weather_calendar.shape)
print("returns_reviews:", returns_reviews.shape)
print("model_ready_daily:", model_ready_daily.shape)

## 11. Create a deliberately messy data-cleaning edition

In [ ]:
raw_model_ready = model_ready_daily.copy()
raw_rng = np.random.default_rng(RANDOM_SEED + 202)

# Add a persistent source identifier before duplicating rows.
raw_model_ready.insert(1, "source_record_id", raw_model_ready["record_id"])

# Approximately 0.8% missingness is introduced independently in selected business fields.
missing_columns = [
    "rainfall_mm", "allocated_ad_spend", "average_rating",
    "competitor_median_price", "average_delivery_days",
]
for column in missing_columns:
    missing_mask = raw_rng.random(len(raw_model_ready)) < 0.008
    raw_model_ready.loc[missing_mask, column] = np.nan

# Inconsistent channel labels emulate manual exports from different source systems.
label_mask = raw_rng.random(len(raw_model_ready)) < 0.006
label_options = {
    "Brand Website": ["brand website", "D2C Web", "Website"],
    "Marketplace": ["market place", "MARKETPLACE", "Marketplace App"],
    "Physical Retail": ["Retail Store", "physical retail", "Offline"],
}
for canonical, alternatives in label_options.items():
    mask = label_mask & raw_model_ready["channel"].eq(canonical)
    if mask.any():
        raw_model_ready.loc[mask, "channel"] = raw_rng.choice(alternatives, mask.sum())

# A small subset uses day-first dates to create a realistic parsing exercise.
raw_model_ready["date"] = pd.to_datetime(raw_model_ready["date"]).dt.strftime("%Y-%m-%d")
mixed_date_mask = raw_rng.random(len(raw_model_ready)) < 0.003
raw_model_ready.loc[mixed_date_mask, "date"] = pd.to_datetime(
    raw_model_ready.loc[mixed_date_mask, "date"]
).dt.strftime("%d/%m/%Y")

# Inventory errors are intentionally small enough to resemble reconciliation mistakes.
reconciliation_mask = raw_rng.random(len(raw_model_ready)) < 0.002
raw_model_ready.loc[reconciliation_mask, "closing_inventory"] += raw_rng.integers(-3, 4, reconciliation_mask.sum())
raw_model_ready["closing_inventory"] = raw_model_ready["closing_inventory"].clip(lower=0)

# Duplicate 0.15% of records and assign new export-level record identifiers.
duplicate_rows = raw_model_ready.sample(frac=0.0015, random_state=RANDOM_SEED).copy()
raw_model_ready = pd.concat([raw_model_ready, duplicate_rows], ignore_index=True)
raw_model_ready["record_id"] = [f"RAW-{i:06d}" for i in range(1, len(raw_model_ready) + 1)]
raw_model_ready = raw_model_ready.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Raw edition rows: {len(raw_model_ready):,}")
print("Missing cells introduced:", int(raw_model_ready.isna().sum().sum()))
print("Duplicate source records:", int(raw_model_ready["source_record_id"].duplicated().sum()))

## 12. Data dictionary and export

In [ ]:
definitions = {
    "record_id": ("Identifier", "Stable synthetic observation identifier.", "None"),
    "date": ("Identifier", "Calendar date of the observation.", "YYYY-MM-DD"),
    "city_id": ("Identifier", "Three-letter fictional market key.", "None"),
    "city": ("Identifier", "Indian metropolitan market.", "None"),
    "channel_id": ("Identifier", "Sales-channel key: D2C, MKT, or RTL.", "None"),
    "channel": ("Identifier", "Brand website, marketplace, or physical retail.", "None"),
    "sku": ("Identifier", "Product colour SKU.", "None"),
    "colour_variant": ("Product", "Colour of the AquaVanta Pulse 750.", "None"),
    "year": ("Calendar", "Calendar year.", "Year"),
    "month": ("Calendar", "Calendar month number.", "1-12"),
    "day_of_week": ("Calendar", "Day name.", "None"),
    "days_since_launch": ("Calendar", "Days elapsed since 1 January 2023.", "Days"),
    "is_weekend": ("Calendar", "Friday, Saturday, or Sunday indicator.", "Boolean"),
    "is_payday_window": ("Calendar", "First five or final four days of a month.", "Boolean"),
    "is_public_holiday": ("Calendar", "Public holiday or modelled festival-window flag.", "Boolean"),
    "festival_name": ("Calendar", "Holi, Eid, Diwali, Christmas, or None.", "None"),
    "maximum_temperature_c": ("Weather", "Daily city-level maximum temperature.", "Degrees Celsius"),
    "rainfall_mm": ("Weather", "Daily city-level rainfall.", "Millimetres"),
    "humidity_pct": ("Weather", "Daily relative humidity.", "Percent"),
    "air_quality_index": ("Weather", "Synthetic daily air-quality index.", "AQI"),
    "heatwave_flag": ("Weather", "Extreme heat or injected heatwave indicator.", "Boolean"),
    "list_price": ("Pricing", "Published AquaVanta product price.", "INR"),
    "discount_pct": ("Pricing", "Discount as a decimal fraction.", "0-1"),
    "selling_price": ("Pricing", "Price after discount.", "INR"),
    "competitor_median_price": ("Competition", "Median simulated competitor price.", "INR"),
    "price_gap_pct": ("Competition", "Relative AquaVanta price gap versus competitor median.", "Decimal fraction"),
    "unit_cost": ("Economics", "Manufacturing and basic fulfilment cost per unit.", "INR"),
    "campaign_type": ("Marketing", "Always-on, influencer, or counter-promotion campaign.", "None"),
    "allocated_ad_spend": ("Marketing", "City-channel budget allocated to this SKU.", "INR"),
    "impressions": ("Marketing", "Paid media impressions allocated to the SKU.", "Count"),
    "clicks": ("Marketing", "Paid-media clicks.", "Count"),
    "product_page_views": ("Funnel", "Online page visits; digital-assisted footfall proxy for retail.", "Count"),
    "add_to_cart_count": ("Funnel", "Add-to-cart actions or strong retail purchase intent.", "Count"),
    "potential_demand": ("Target", "Demand before cancellations and inventory constraints.", "Units"),
    "units_ordered": ("Target", "Customer units ordered; equal to potential demand in this edition.", "Units"),
    "cancellation_units": ("Outcome", "Units cancelled before fulfilment.", "Units"),
    "units_sold": ("Primary target", "Units fulfilled after cancellations and stock constraints.", "Units"),
    "lost_sales_units": ("Outcome", "Unfulfilled demand caused by insufficient stock.", "Units"),
    "conversion_rate": ("Funnel", "Potential demand divided by product-page views.", "Decimal fraction"),
    "opening_inventory": ("Inventory", "Stock before receipts and sales.", "Units"),
    "replenishment_units": ("Inventory", "Units received from earlier purchase orders.", "Units"),
    "order_placed_units": ("Inventory", "Replenishment order placed on the date.", "Units"),
    "closing_inventory": ("Inventory", "Stock after receipts and fulfilled sales.", "Units"),
    "stockout_hours": ("Inventory", "Estimated hours unavailable during the day.", "Hours"),
    "inventory_cover_days": ("Inventory", "Closing stock divided by lagged 14-day average sales.", "Days"),
    "supplier_lead_time_days": ("Inventory", "Modelled replenishment lead time when ordering.", "Days"),
    "average_delivery_days": ("Delivery", "Average time from fulfilment to customer delivery.", "Days"),
    "return_units": ("Outcome", "Units returned from purchases made approximately 14 days earlier.", "Units"),
    "return_rate": ("Outcome", "Return units divided by lagged eligible sales.", "Decimal fraction"),
    "new_review_count": ("Feedback", "Reviews linked to purchases approximately 10 days earlier.", "Count"),
    "average_rating": ("Feedback", "Average rating among new reviews; missing when no review is posted.", "1-5"),
    "negative_review_share": ("Feedback", "Share of new reviews rated one or two stars.", "Decimal fraction"),
    "primary_return_reason": ("Feedback", "Dominant return reason for the observation.", "None"),
    "gross_revenue": ("Economics", "Current fulfilled sales multiplied by selling price.", "INR"),
    "return_value": ("Economics", "Value refunded for returns from earlier purchases.", "INR"),
    "net_revenue": ("Economics", "Gross revenue minus return value.", "INR"),
    "gross_margin": ("Economics", "Net revenue less product cost, fees, marketing, and return handling.", "INR"),
    "event_name": ("Event", "Injected event label; multiple overlapping events are pipe-separated.", "None"),
}

data_dictionary = pd.DataFrame(
    [
        {
            "column_name": column,
            "data_type": str(model_ready_daily[column].dtype),
            "category": definitions.get(column, ("Other", "", ""))[0],
            "description": definitions.get(column, ("Other", "Synthetic model field.", "None"))[1],
            "unit_or_format": definitions.get(column, ("Other", "", "None"))[2],
        }
        for column in model_ready_daily.columns
    ]
)

# Write all deliverables with explicit filenames so rerunning the notebook is idempotent.
sales_daily.to_csv(CLEAN_DIR / "sales_daily.csv", index=False, date_format="%Y-%m-%d")
inventory_daily.to_csv(CLEAN_DIR / "inventory_daily.csv", index=False, date_format="%Y-%m-%d")
marketing_daily.to_csv(CLEAN_DIR / "marketing_daily.csv", index=False, date_format="%Y-%m-%d")
weather_calendar.to_csv(CLEAN_DIR / "weather_calendar.csv", index=False, date_format="%Y-%m-%d")
returns_reviews.to_csv(CLEAN_DIR / "returns_reviews.csv", index=False, date_format="%Y-%m-%d")
events.to_csv(CLEAN_DIR / "market_events.csv", index=False, date_format="%Y-%m-%d")
model_ready_daily.to_csv(CLEAN_DIR / "model_ready_daily.csv", index=False, date_format="%Y-%m-%d")
raw_model_ready.to_csv(RAW_DIR / "model_ready_daily_raw.csv", index=False)
data_dictionary.to_csv(METADATA_DIR / "data_dictionary.csv", index=False)

print("Exported files:")
for path in sorted(OUTPUT_ROOT.rglob("*.csv")):
    print(f"  {path} ({path.stat().st_size / 1_000_000:.2f} MB)")

## 13. Validation and integrity report

In [ ]:
validation_checks = {
    "expected_row_count": len(model_ready_daily) == 105_216,
    "unique_business_grain": not model_ready_daily.duplicated(["date", "city_id", "channel_id", "sku"]).any(),
    "complete_date_range": model_ready_daily["date"].min() == pd.Timestamp(START_DATE) and model_ready_daily["date"].max() == pd.Timestamp(END_DATE),
    "nonnegative_sales": model_ready_daily["units_sold"].ge(0).all(),
    "sales_do_not_exceed_fulfillable_orders": (model_ready_daily["units_sold"] <= model_ready_daily["units_ordered"] - model_ready_daily["cancellation_units"]).all(),
    "funnel_is_monotonic": (
        (model_ready_daily["clicks"] <= model_ready_daily["impressions"])
        & (model_ready_daily["add_to_cart_count"] <= model_ready_daily["product_page_views"])
        & (model_ready_daily["potential_demand"] <= model_ready_daily["add_to_cart_count"])
    ).all(),
    "inventory_conservation": (
        model_ready_daily["opening_inventory"]
        + model_ready_daily["replenishment_units"]
        - model_ready_daily["units_sold"]
        == model_ready_daily["closing_inventory"]
    ).all(),
    "discount_range": model_ready_daily["discount_pct"].between(0, 0.30).all(),
    "rating_range": model_ready_daily["average_rating"].dropna().between(1, 5).all(),
    "clean_identifiers_complete": model_ready_daily[identifier_cols].notna().all().all(),
}

assert all(validation_checks.values()), {k: v for k, v in validation_checks.items() if not v}

main_csv = CLEAN_DIR / "model_ready_daily.csv"
sha256 = hashlib.sha256(main_csv.read_bytes()).hexdigest()
annual_summary = (
    model_ready_daily.groupby("year", as_index=False)
    .agg(
        units_sold=("units_sold", "sum"),
        potential_demand=("potential_demand", "sum"),
        net_revenue=("net_revenue", "sum"),
        gross_margin=("gross_margin", "sum"),
        lost_sales_units=("lost_sales_units", "sum"),
    )
)

generation_summary = {
    "dataset_name": "AquaVanta Pulse 750 Synthetic Omnichannel Sales",
    "fictional_data": True,
    "random_seed": RANDOM_SEED,
    "date_range": [START_DATE, END_DATE],
    "clean_row_count": int(len(model_ready_daily)),
    "raw_row_count": int(len(raw_model_ready)),
    "cities": int(model_ready_daily["city_id"].nunique()),
    "channels": int(model_ready_daily["channel_id"].nunique()),
    "variants": int(model_ready_daily["sku"].nunique()),
    "total_units_sold": int(model_ready_daily["units_sold"].sum()),
    "stockout_row_share": round(float(model_ready_daily["stockout_hours"].gt(0).mean()), 6),
    "main_csv_sha256": sha256,
    "validation_checks": {key: bool(value) for key, value in validation_checks.items()},
}

(METADATA_DIR / "generation_summary.json").write_text(
    json.dumps(generation_summary, indent=2), encoding="utf-8"
)
annual_summary.to_csv(METADATA_DIR / "annual_summary.csv", index=False)

print(pd.Series(validation_checks, name="passed").to_string())
print("\nAnnual summary:")
print(annual_summary.to_string(index=False))
print("\nSHA-256:", sha256)

## 14. Visual QA: expected real-world chart patterns

In [ ]:
# Chart 1: monthly sales should grow over time while retaining summer and festival peaks.
monthly_sales = (
    model_ready_daily.assign(month_start=model_ready_daily["date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month_start", as_index=False)["units_sold"].sum()
)
fig, ax = plt.subplots(figsize=(13, 4.5))
sns.lineplot(data=monthly_sales, x="month_start", y="units_sold", marker="o", ax=ax, color="#1677B8")
ax.set(title="Monthly AquaVanta Pulse 750 Sales", xlabel="Month", ylabel="Units sold")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_monthly_sales_trend.png", dpi=160, bbox_inches="tight")
plt.show()

# Chart 2: normalize within city, channel, SKU, and year before comparing temperature.
# This prevents large cold-weather cities such as Delhi from creating a misleading raw
# average and exposes the intended within-market hydration-demand relationship.
temperature_chart_data = model_ready_daily.loc[
    model_ready_daily["event_name"].eq("No event")
    & model_ready_daily["festival_name"].eq("No festival")
    & model_ready_daily["days_since_launch"].gt(60)
].copy()
temperature_chart_data["group_baseline_demand"] = temperature_chart_data.groupby(
    ["city_id", "channel_id", "sku", "year"]
)["potential_demand"].transform("mean")
temperature_chart_data["demand_index"] = (
    100 * temperature_chart_data["potential_demand"] / temperature_chart_data["group_baseline_demand"]
)
temperature_chart_data["temperature_band"] = pd.cut(
    temperature_chart_data["maximum_temperature_c"],
    bins=[-np.inf, 20, 25, 30, 35, 40, np.inf],
    labels=["<20", "20-25", "25-30", "30-35", "35-40", "40+"],
)
temperature_effect = temperature_chart_data.groupby(
    "temperature_band", observed=True, as_index=False
)["demand_index"].mean()
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=temperature_effect, x="temperature_band", y="demand_index", ax=ax, color="#35A7A0")
ax.axhline(100, color="#444444", linewidth=1, linestyle="--")
ax.set(title="Temperature Effect on Normalized Demand", xlabel="Temperature band (°C)", ylabel="Demand index (group mean = 100)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_temperature_demand.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 3: conversion is aligned with today's discount, whereas returns are aligned with
# the discount on the originating purchase 14 days earlier. This respects the simulation's
# delayed-return design instead of accidentally comparing returns with today's promotion.
discount_bins = [-0.001, 0.05, 0.10, 0.15, 0.20, 0.25, 0.31]
discount_labels = ["0-5%", "5-10%", "10-15%", "15-20%", "20-25%", "25-30%"]
discount_chart_data = model_ready_daily.copy()
discount_chart_data["purchase_discount_14_days_ago"] = discount_chart_data.groupby(
    ["city_id", "channel_id", "sku"]
)["discount_pct"].shift(14)
discount_chart_data["discount_band"] = pd.cut(
    discount_chart_data["discount_pct"], bins=discount_bins, labels=discount_labels
)
discount_chart_data["purchase_discount_band"] = pd.cut(
    discount_chart_data["purchase_discount_14_days_ago"], bins=discount_bins, labels=discount_labels
)
conversion_by_discount = (
    discount_chart_data.groupby("discount_band", observed=True, as_index=False)["conversion_rate"].mean()
)
returns_by_purchase_discount = (
    discount_chart_data.groupby("purchase_discount_band", observed=True, as_index=False)["return_rate"].mean()
    .rename(columns={"purchase_discount_band": "discount_band"})
)
discount_effect = conversion_by_discount.merge(
    returns_by_purchase_discount, on="discount_band", how="left", validate="one_to_one"
)
fig, ax1 = plt.subplots(figsize=(9, 4.8))
ax2 = ax1.twinx()
sns.lineplot(data=discount_effect, x="discount_band", y="conversion_rate", marker="o", ax=ax1, color="#1677B8", label="Conversion")
sns.lineplot(data=discount_effect, x="discount_band", y="return_rate", marker="s", ax=ax2, color="#D95F59", label="Return rate")
ax1.set(title="Promotion Trade-off: Conversion versus Lagged Returns", xlabel="Purchase discount band", ylabel="Conversion rate")
ax2.set_ylabel("Return rate")
ax2.text(0.99, 0.02, "Returns linked to purchases 14 days earlier", transform=ax2.transAxes, ha="right", va="bottom", fontsize=9)
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_discount_tradeoff.png", dpi=160, bbox_inches="tight")
plt.show()

# Chart 4: marketplace leads volume, while channel economics differ after fees and returns.
channel_summary = (
    model_ready_daily.groupby("channel", as_index=False)
    .agg(units_sold=("units_sold", "sum"), gross_margin=("gross_margin", "sum"))
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=channel_summary, x="channel", y="units_sold", ax=axes[0], color="#6A8EAE")
sns.barplot(data=channel_summary, x="channel", y="gross_margin", ax=axes[1], color="#7BAE7F")
axes[0].set(title="Sales Volume by Channel", xlabel="", ylabel="Units sold")
axes[1].set(title="Gross Margin by Channel", xlabel="", ylabel="INR")
for ax in axes:
    ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_channel_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 5: fulfilled sales flatten under long stockouts even when potential demand remains high.
stockout_effect = (
    model_ready_daily.assign(
        stockout_band=pd.cut(
            model_ready_daily["stockout_hours"],
            bins=[-0.01, 0.01, 4, 8, 16, 24.01],
            labels=["None", "0-4h", "4-8h", "8-16h", "16-24h"],
        )
    )
    .groupby("stockout_band", observed=True, as_index=False)
    .agg(potential_demand=("potential_demand", "mean"), units_sold=("units_sold", "mean"), lost_sales=("lost_sales_units", "mean"))
)
stockout_long = stockout_effect.melt(
    id_vars="stockout_band",
    value_vars=["potential_demand", "units_sold", "lost_sales"],
    var_name="measure",
    value_name="average_units",
)
fig, ax = plt.subplots(figsize=(10, 4.8))
sns.barplot(data=stockout_long, x="stockout_band", y="average_units", hue="measure", ax=ax)
ax.set(title="Demand Censoring during Stockouts", xlabel="Stockout duration", ylabel="Average units")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_stockout_censoring.png", dpi=160, bbox_inches="tight")
plt.show()

# Chart 6: the quality issue should create a delayed return spike for Coral and Sage.
defect_window = model_ready_daily["date"].between("2024-07-01", "2024-11-30")
quality_timeline = (
    model_ready_daily.loc[defect_window]
    .assign(month_start=lambda x: x["date"].dt.to_period("M").dt.to_timestamp())
    .groupby(["month_start", "colour_variant"], as_index=False)["return_units"].sum()
)
fig, ax = plt.subplots(figsize=(10, 4.8))
sns.lineplot(data=quality_timeline, x="month_start", y="return_units", hue="colour_variant", marker="o", ax=ax)
ax.set(title="Delayed Returns after the 2024 Battery-cap Issue", xlabel="Month", ylabel="Return units")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "06_quality_event_returns.png", dpi=160, bbox_inches="tight")
plt.show()

print(f"Saved {len(list(FIGURE_DIR.glob('*.png')))} QA figures to {FIGURE_DIR.resolve()}")